# nb07 - Gold layer + serving the dashboard

Gold answers business questions with joins already resolved: one row per grain, ready
to serve. This notebook builds exactly the tables the **Streamlit dashboard**
(http://localhost:8501) consumes - the contract is listed at the bottom.

**You will learn:** dimensional joins on a star-ish schema, metric definitions in SQL,
`CREATE OR REPLACE TABLE` (idempotent gold rebuilds), and serving-path considerations
(how often to rebuild vs. what the dashboard caches).

**Prerequisites:** nb02 (silver orders/customers/products) and the CSV dimension from
bronze. nb05 is optional - without it the dashboard's Streaming tab shows a hint.


In [1]:
from pyspark.sql import functions as F

from de_stack import get_spark, BRONZE, register_table

spark = get_spark("nb07-gold", shuffle_partitions=8)

for name in ("orders", "customers", "products"):
    if not register_table(spark, "silver", name):
        raise RuntimeError(f"silver.{name} missing - run nb02 first")

# small CSV dimension: country -> region (+ a tax multiplier we do not need here)
(spark.read.option("header", True).csv(f"{BRONZE}/country_region.csv")
    .createOrReplaceTempView("country_region"))

spark.sql("CREATE DATABASE IF NOT EXISTS gold")
print("gold database ready")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 11:41:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/10/13 11:41:38 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.
26/10/13 11:41:38 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.


gold database ready


## 1. The enrichment spine

One join resolves country -> region and product -> category. Every metric below reads
from this temp view, so grain rules live in exactly one place.


In [2]:
spark.sql('''
    CREATE OR REPLACE TEMP VIEW enriched AS
    SELECT o.order_id,
           o.order_date,
           o.status,
           o.quantity,
           o.total_amount,
           c.country,
           r.region,
           p.product_id,
           p.name    AS product_name,
           p.category
    FROM silver.orders   o
    JOIN silver.customers c ON o.customer_id = c.customer_id
    JOIN country_region   r ON c.country     = r.country
    JOIN silver.products  p ON o.product_id  = p.product_id
''')

spark.sql("SELECT * FROM enriched LIMIT 5").show(truncate=False)
print("enriched rows:", spark.sql("SELECT COUNT(*) FROM enriched").first()[0])


26/10/13 11:41:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+----------+---------+--------+------------+-------+------+----------+--------------+--------+
|order_id|order_date|status   |quantity|total_amount|country|region|product_id|product_name  |category|
+--------+----------+---------+--------+------------+-------+------+----------+--------------+--------+
|O900444 |2026-10-07|cancelled|1       |106.01      |DE     |EMEA  |P10237    |Staff Send    |home    |
|O900476 |2026-10-07|paid     |1       |30.45       |BR     |LATAM |P10058    |Enjoy Decision|books   |
|O900885 |2026-10-07|cancelled|2       |43.82       |AU     |APAC  |P10297    |Education Tv  |grocery |
|O901126 |2026-10-07|created  |3       |58.83       |IN     |APAC  |P10031    |Age Scene     |beauty  |
|O901349 |2026-10-07|delivered|1       |15.24       |GB     |EMEA  |P10443    |Morning Party |grocery |
+--------+----------+---------+--------+------------+-------+------+----------+--------------+--------+



enriched rows: 58847


## 2. The metric tables

`CREATE OR REPLACE TABLE ... AS SELECT` makes gold **idempotent**: re-running this
notebook rebuilds every table from scratch instead of appending duplicates. Note the
grain of each: per-day, per-region, per-category, per-product, per-status.

`COUNT(total_amount)` (not `COUNT(*)`) keeps the denominator consistent with revenue -
dirty rows whose `try_cast` produced NULL amount are excluded from both.


In [3]:
spark.sql('''
    CREATE OR REPLACE TABLE gold.orders_daily USING DELTA AS
    SELECT order_date,
           ROUND(SUM(total_amount), 2)                    AS revenue,
           COUNT(total_amount)                            AS orders,
           ROUND(SUM(total_amount) / COUNT(total_amount), 2) AS aov
    FROM enriched
    WHERE order_date IS NOT NULL
    GROUP BY order_date
''')

spark.sql('''
    CREATE OR REPLACE TABLE gold.revenue_by_region USING DELTA AS
    SELECT region,
           ROUND(SUM(total_amount), 2) AS revenue,
           COUNT(total_amount)         AS orders
    FROM enriched
    GROUP BY region
''')

spark.sql('''
    CREATE OR REPLACE TABLE gold.revenue_by_category USING DELTA AS
    SELECT category,
           ROUND(SUM(total_amount), 2) AS revenue,
           COUNT(total_amount)         AS orders
    FROM enriched
    GROUP BY category
''')

spark.sql('''
    CREATE OR REPLACE TABLE gold.order_status_funnel USING DELTA AS
    SELECT status, COUNT(*) AS orders
    FROM enriched
    GROUP BY status
''')

print("4 gold tables written")


4 gold tables written


In [4]:
spark.sql('''
    CREATE OR REPLACE TABLE gold.top_products USING DELTA AS
    SELECT product_id,
           product_name AS name,
           category,
           ROUND(SUM(total_amount), 2) AS revenue,
           COALESCE(SUM(quantity), 0)   AS units
    FROM enriched
    GROUP BY product_id, product_name, category
''')

print("gold.top_products written:",
      spark.sql("SELECT COUNT(*) FROM gold.top_products").first()[0], "products")


gold.top_products written: 500 products


## 3. Streaming metrics (optional - runs only after nb05)

If nb05 populated `silver.events`, we publish per-minute event windows. Otherwise this
cell reports the gap and the dashboard simply hides the detail.


In [5]:
if register_table(spark, "silver", "events") and spark.sql("SELECT COUNT(*) FROM silver.events").first()[0] > 0:
    spark.sql('''
        CREATE OR REPLACE TABLE gold.events_funnel USING DELTA AS
        SELECT window(event_ts, '1 minute').start AS window_start,
               event_type,
               COUNT(*)                           AS events
        FROM silver.events
        GROUP BY window(event_ts, '1 minute'), event_type
    ''')
    print("gold.events_funnel written:",
          spark.sql("SELECT COUNT(*) FROM gold.events_funnel").first()[0], "windows")
else:
    print("silver.events is empty - run nb05, then re-run this cell (dashboard shows a hint meanwhile)")


26/10/13 11:42:37 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.


gold.events_funnel written: 172 windows


## 4. Verify what the dashboard will see


In [6]:
for t in ("orders_daily", "revenue_by_region", "revenue_by_category",
          "top_products", "order_status_funnel"):
    df = spark.sql(f"SELECT * FROM gold.{t}")
    print(f"gold.{t:24s} {df.count():6d} rows")
    df.orderBy(df.columns[0]).show(3, truncate=40)

# sanity: totals agree between the daily and regional cuts
daily = spark.sql("SELECT ROUND(SUM(revenue), 2) FROM gold.orders_daily").first()[0]
region = spark.sql("SELECT ROUND(SUM(revenue), 2) FROM gold.revenue_by_region").first()[0]
assert daily == region, (daily, region)
print("cross-check OK: daily revenue == regional revenue:", daily)


gold.orders_daily                 31 rows


+----------+---------+------+-----+
|order_date|  revenue|orders|  aov|
+----------+---------+------+-----+
|2026-09-13| 69431.45|   929|74.74|
|2026-09-14|143626.35|  1917|74.92|
|2026-09-15| 140536.9|  1987|70.73|
+----------+---------+------+-----+
only showing top 3 rows


gold.revenue_by_region             4 rows


+------+----------+------+
|region|   revenue|orders|
+------+----------+------+
|  APAC| 1275748.7| 17564|
|  EMEA|2156985.56| 29875|
| LATAM| 433952.78|  5865|
+------+----------+------+
only showing top 3 rows


gold.revenue_by_category           8 rows


...

gold.top_products                500 rows


+----------+---------------+-----------+-------+-----+
|product_id|           name|   category|revenue|units|
+----------+---------------+-----------+-------+-----+
|    P10000|     Might Bank|    grocery|3395.28|  226|
|    P10001|Analysis Remain|electronics|2339.46|  239|
|    P10002|     None Those|    fashion|2775.51|  241|
+----------+---------------+-----------+-------+-----+
only showing top 3 rows


gold.order_status_funnel           6 rows


+---------+------+
|   status|orders|
+---------+------+
|cancelled|  2943|
|  created|  4610|
|delivered| 14785|
+---------+------+
only showing top 3 rows


cross-check OK: daily revenue == regional revenue: 4268144.43


## 5. The serving contract

Open **http://localhost:8501** - the dashboard reads exactly these tables (30s cache,
`Refresh data` button in the sidebar):

| table | columns the app expects |
|-------|-------------------------|
| `gold.orders_daily` | `order_date, revenue, orders, aov` |
| `gold.revenue_by_region` | `region, revenue, orders` |
| `gold.revenue_by_category` | `category, revenue, orders` |
| `gold.top_products` | `product_id, name, category, revenue, units` |
| `gold.order_status_funnel` | `status, orders` |
| `gold.events_funnel` | `window_start, event_type, events` (optional) |

Production note: this notebook is a *batch rebuild*. A real serving tier would either
schedule it (cron/Orchestrator), turn each metric into an incremental `MERGE` (nb04's
pattern), or materialize from the stream directly (nb05's `foreachBatch`). The dashboard
code would not change either way - that is the point of a stable gold contract.

**Course complete.** Revisit any notebook freely: they are idempotent, and
`make reset && make seed` returns the lake to a known state.
